# Apex3D: fine-grained car classifier (Stanford Cars, 196 classes)

Trains a **ConvNeXt-Tiny** on Stanford Cars in Google Colab, evaluates it, calibrates its confidence, and exports it for the
FastAPI backend. Every number in the output comes from this notebook's own run, and nothing is typed in by hand.

**How to run**
1. `Runtime > Change runtime type > T4 GPU`.
2. Add your Kaggle API key as Colab *Secrets* named `KAGGLE_USERNAME` and `KAGGLE_KEY` (key icon in the left bar). If you don't, the notebook asks you to upload `kaggle.json` instead. The key is never printed or saved to Drive.
3. `Runtime > Run all`. The first cells check the dataset and stop with a clear message if something looks wrong.
4. If Colab disconnects, run all again. Training resumes from the last saved epoch because checkpoints go to your Google Drive.

**What this notebook covers:** dataset check, stratified 85/15 train/validation split, transfer learning with staged fine-tuning, Top-1 and Top-5 on the
unseen test set (ground-truth crops and detector crops), per-class precision/recall/F1, confusion matrix and most-confused pairs, temperature scaling and
confidence-threshold analysis, and an optimisation step (ONNX, INT8 and half precision) with size, speed and accuracy before and after.

**Not covered here** (Person B's notebooks): the classical baseline, hybrid fusion, the ablation study, robustness tests, and detector IoU and mAP.

**Dataset licence:** Stanford Cars is generally described as for non-commercial research. Treat Apex3D as a research demonstration.

In [ ]:
# 1. Setup ------------------------------------------------------------------------------------
%pip install -q timm onnx onnxruntime onnxscript kagglehub

import os, sys, json, time, math, random, platform, shutil, hashlib, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore", category=UserWarning)

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("APEX_SMOKE") == "1"   # a tiny run used only to test this notebook itself. Never set it for real training.

CFG = dict(
    seed=42,
    model_name="convnext_tiny.fb_in22k_ft_in1k",
    img_size=224,            # matches backend/services/classifier_service.py, which resizes crops to 224x224
    batch_size=64,
    num_workers=2,           # Colab gives 2 CPU cores
    drop_path=0.1,
    head_epochs=3,           # stage 1: only the new classifier head trains
    head_lr=1e-3,
    full_epochs=22,          # stage 2: everything trains, with layer-wise learning-rate decay
    full_lr=3e-4,
    layer_decay=0.75,
    weight_decay=0.05,
    warmup_epochs=1,
    label_smoothing=0.1,
    patience=6,              # early stopping on validation Top-1
    val_fraction=0.15,
    bbox_margin=0.10,        # extra context around the ground-truth box
    cache_short_side=288,    # crops are cached at this size so training isn't limited by JPEG decoding
    target_kept_accuracy=0.90,
    quant_eval_n=1000,       # test images used to compare fp32 and INT8 accuracy on CPU (CPU inference is slow)
    bench_runs=30,
    run_detector_crop_eval=True,
)
if SMOKE:
    CFG.update(img_size=128, batch_size=16, num_workers=0, head_epochs=int(os.environ.get("APEX_SMOKE_HEAD", 1)), full_epochs=int(os.environ.get("APEX_SMOKE_FULL", 1)),
               val_fraction=0.30, quant_eval_n=40, bench_runs=3, run_detector_crop_eval=os.environ.get("APEX_SMOKE_DET") == "1", cache_short_side=160)

random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Where results go. On Colab this is your Google Drive, so a disconnect never loses work.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/apex3d")
else:
    WORK = Path(os.environ.get("APEX_WORK", "./apex3d_out")).resolve()
ART = WORK / "artifacts"
CKPT = WORK / "checkpoints"
for p in (WORK, ART, CKPT):
    p.mkdir(parents=True, exist_ok=True)
CACHE = Path("/content/cache") if IN_COLAB else WORK / "cache"
CACHE.mkdir(parents=True, exist_ok=True)

print(f"device       : {device}" + (f" ({torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)" if device.type == "cuda" else ""))
print(f"results in   : {WORK}")
print(f"python/torch : {platform.python_version()} / {torch.__version__}")
if device.type != "cuda" and not SMOKE:
    raise SystemExit("No GPU found. Use Runtime > Change runtime type > T4 GPU, then run all again.")
if SMOKE:
    print("*** SMOKE TEST MODE: tiny run, results are meaningless ***")

## 2. Get the dataset

Downloads Stanford Cars from Kaggle. The official page and torchvision's download link are both broken, and the mirror named in the project proposal
(`jessicali9530/stanford-cars-dataset`) has since been removed from Kaggle. So the notebook tries the copies listed in `SLUGS`, in order, and uses the first one
whose layout it can read. They are all the same dataset, not a substitute, and nothing else is used for training.

If none work, download Stanford Cars yourself, put the folder in Google Drive, and set `MANUAL_DATA_DIR` in the next cell.

In [ ]:
# 2. Download -----------------------------------------------------------------------------------
def ensure_kaggle_credentials():
    """Uses existing credentials, Colab secrets, or an uploaded kaggle.json. Never prints the key."""
    if os.environ.get("KAGGLE_API_TOKEN") or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        return
    kaggle_dir = Path.home() / ".kaggle"
    kj = kaggle_dir / "kaggle.json"
    if kj.exists() or (kaggle_dir / "access_token").exists():
        return
    if IN_COLAB:
        from google.colab import userdata

        def secret(name):
            try:
                return userdata.get(name)
            except Exception as e:      # say why, so a wrong name or a switched-off toggle is obvious
                print(f"  secret {name}: {type(e).__name__}: {str(e)[:160]}")
                return None

        print("Looking for Kaggle credentials in Colab secrets...")
        user, key, token = secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY"), secret("KAGGLE_API_TOKEN")
        if user and key:
            os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user.strip(), key.strip()
            print("  using the KAGGLE_USERNAME and KAGGLE_KEY secrets")
            return
        if token:
            os.environ["KAGGLE_API_TOKEN"] = token.strip()
            print("  using the KAGGLE_API_TOKEN secret")
            return
        print("  No usable secrets. Click the key icon in the left bar and check that the names are exactly KAGGLE_USERNAME and KAGGLE_KEY, "
              "and that the 'Notebook access' switch is ON for each. Then run this cell again.")
        from google.colab import files
        print("Or upload your kaggle.json (Kaggle > Settings > API > Create New Token):")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise SystemExit("No Kaggle credentials. Fix the secrets (see above) and run this cell again.")
        kj.parent.mkdir(exist_ok=True)
        kj.write_bytes(uploaded["kaggle.json"])
        kj.chmod(0o600)
        return
    raise SystemExit("Set KAGGLE_USERNAME and KAGGLE_KEY, or put kaggle.json in ~/.kaggle/.")


SLUGS = [
    "jutrera/stanford-car-dataset-by-classes-folder",
    "rickyyyyyyy/torchvision-stanford-cars",
    "eduardo4jesus/stanford-cars-dataset",
]
MANUAL_DATA_DIR = ""   # optional: a folder you downloaded yourself, e.g. "/content/drive/MyDrive/stanford_cars". If set, nothing is downloaded.


def download(slug):
    ensure_kaggle_credentials()
    import kagglehub
    return Path(kagglehub.dataset_download(slug))


def show_tree(root, depth=3, limit=14):
    root = Path(root)
    def walk(p, d):
        kids = sorted(p.iterdir(), key=lambda x: (x.is_file(), x.name.lower()))
        dirs = [k for k in kids if k.is_dir()]
        files = [k for k in kids if k.is_file()]
        for k in dirs[:limit]:
            n = sum(1 for _ in k.rglob("*") if _.is_file())
            print("  " * d + f"{k.name}/  ({n} files)")
            if d + 1 < depth:
                walk(k, d + 1)
        if len(dirs) > limit:
            print("  " * d + f"... and {len(dirs) - limit} more folders")
        for f in files[:limit]:
            print("  " * d + f"{f.name}  ({f.stat().st_size/1e3:.0f} KB)")
    walk(root, 0)

In [ ]:
# 3. Read the annotations (works with the Kaggle CSV layout and the official .mat layout) --------------
IMG_EXT = (".jpg", ".jpeg", ".png")


def find_one(root, name):
    hits = sorted(Path(root).rglob(name))
    return hits[0] if hits else None


def index_images(root, split):
    """Maps file name -> path. The Kaggle mirror has flat folders (cars_train) and per-class folders (car_data/train/<class>).
    The per-class copy is preferred, because its folder name lets us check the labels."""
    wanted = {split, f"cars_{split}"}
    out, rank = {}, {}
    for d in Path(root).rglob("*"):
        if d.is_dir() and d.name.lower() in wanted:
            for f in d.rglob("*"):
                if f.suffix.lower() in IMG_EXT:
                    r = 0 if f.parent.name.lower() not in wanted else 1
                    if f.name not in out or r < rank[f.name]:
                        out[f.name], rank[f.name] = f, r
    return out


def read_anno_csv(path):
    raw = pd.read_csv(path, header=None)
    if not str(raw.iloc[0, 0]).lower().endswith(IMG_EXT):   # the file has a header row
        raw = pd.read_csv(path)
    if raw.shape[1] >= 6:
        raw = raw.iloc[:, :6]
        raw.columns = ["fname", "x1", "y1", "x2", "y2", "cls"]
    elif raw.shape[1] == 2:
        raw.columns = ["fname", "cls"]
        for c in ("x1", "y1", "x2", "y2"):
            raw[c] = np.nan
    else:
        raise ValueError(f"Unexpected annotation columns in {path}: {list(raw.columns)}")
    return raw


def load_class_folders(root):
    """Copies that are only train/ and test/ folders with one sub-folder per car class and no annotation files. There are no boxes, so whole images are used."""
    rows = []
    for split in ("train", "test"):
        for d in Path(root).rglob("*"):
            if d.is_dir() and d.name.lower() in (split, f"cars_{split}"):
                for f in d.rglob("*"):
                    if f.suffix.lower() in IMG_EXT and f.parent != d:
                        rows.append((split, f.parent.name, f))
    if not rows:
        raise SystemExit("Could not recognise the dataset layout: no annotation files and no per-class folders.")
    raw = pd.DataFrame(rows, columns=["split", "cls_name", "path"]).drop_duplicates(["split", "path"])
    names = sorted(raw[raw.split == "train"].cls_name.unique())
    if len(names) != 196:
        raise SystemExit(f"Found {len(names)} class folders in train, expected 196.")
    if sum(n.lower().startswith("class_") or n.isdigit() for n in names) > 100:
        raise SystemExit("The class folders are numbered (class_001, ...), not named. The app needs real make, model and year names, so this copy cannot be used.")
    lookup = {n.lower(): i for i, n in enumerate(names)}
    raw = raw[raw.cls_name.str.lower().isin(lookup)].copy()
    raw["class_id"] = raw.cls_name.str.lower().map(lookup)
    raw["fname"] = raw.path.map(lambda f: f.name)
    for c in ("x1", "y1", "x2", "y2"):
        raw[c] = np.nan
    return raw[["fname", "x1", "y1", "x2", "y2", "class_id", "split", "path"]].reset_index(drop=True), names, "class-folders"


def load_stanford_cars(root):
    root = Path(root)
    a_tr, a_te, names_csv = find_one(root, "anno_train.csv"), find_one(root, "anno_test.csv"), find_one(root, "names.csv")
    if a_tr and a_te and names_csv:   # Kaggle CSV layout
        layout = "kaggle-csv"
        names = pd.read_csv(names_csv, header=None)[0].astype(str).str.strip().tolist()
        if names[0].lower() in ("class", "name", "names", "class_name"):
            names = names[1:]
        frames = []
        for split, a in (("train", a_tr), ("test", a_te)):
            df = read_anno_csv(a)
            df["split"] = split
            frames.append(df)
        df = pd.concat(frames, ignore_index=True)
        if df["cls"].map(lambda v: isinstance(v, str) and not str(v).isdigit()).any():
            lookup = {n.lower(): i for i, n in enumerate(names)}
            df["class_id"] = df["cls"].map(lambda v: lookup[str(v).strip().lower()])
        else:
            df["class_id"] = df["cls"].astype(int) - (1 if df["cls"].astype(int).min() == 1 else 0)
    else:
        mat_meta = find_one(root, "cars_meta.mat")
        mat_tr = find_one(root, "cars_train_annos.mat")
        mat_te = find_one(root, "cars_test_annos_withlabels.mat")
        if not (mat_meta and mat_tr):
            return load_class_folders(root)
        if not mat_te:
            raise SystemExit("The test labels file (cars_test_annos_withlabels.mat) is missing. The original devkit's test set has no labels, "
                             "so test accuracy cannot be measured. Use the Kaggle mirror instead.")
        from scipy.io import loadmat
        layout = "official-mat"
        names = [str(n[0]) for n in loadmat(mat_meta, squeeze_me=False)["class_names"][0]]
        frames = []
        for split, m in (("train", mat_tr), ("test", mat_te)):
            ann = loadmat(m, squeeze_me=True, struct_as_record=False)["annotations"]
            frames.append(pd.DataFrame(dict(
                fname=[str(a.fname) for a in ann], x1=[a.bbox_x1 for a in ann], y1=[a.bbox_y1 for a in ann],
                x2=[a.bbox_x2 for a in ann], y2=[a.bbox_y2 for a in ann], class_id=[int(a.__getattribute__("class")) - 1 for a in ann], split=split)))
        df = pd.concat(frames, ignore_index=True)

    paths = {s: index_images(root, s) for s in ("train", "test")}
    df["path"] = [paths[s].get(f) for s, f in zip(df["split"], df["fname"])]
    return df, names, layout


local = os.environ.get("APEX_DATA_ROOT") or MANUAL_DATA_DIR
candidates = [("your own folder", None)] if local else [(slug, slug) for slug in SLUGS]
df = None
for name, slug in candidates:
    try:
        root = Path(local) if slug is None else download(slug)
        print(f"\n=== {name}\n    at {root}")
        show_tree(root)
        df, class_names, layout = load_stanford_cars(root)
        if df.path.notna().mean() < 0.99 or df.split.nunique() != 2:
            raise ValueError(f"only {int(df.path.notna().sum())} of {len(df)} images were found")
        DATA_ROOT = root
        break
    except (Exception, SystemExit) as e:          # a removed dataset, a 403, or a layout we cannot read: try the next one
        print(f"    not usable: {type(e).__name__}: {str(e)[:300]}")
        df = None
if df is None:
    raise SystemExit("None of the datasets could be used (see the messages above). Download Stanford Cars yourself, put the folder in Google Drive, "
                     "set MANUAL_DATA_DIR in the download cell, and run again.")
print(f"\nUsing: {name}   layout: {layout}   classes: {len(class_names)}   rows: {len(df)}")
display(df.head(3))

In [ ]:
# 4. Check the data before trusting it. Test labels from a mirror are a third party's work, so we verify them. --------------------
def norm_name(s):
    return "".join(ch for ch in str(s).lower() if ch.isalnum())

checks = []   # (name, ok, detail, critical)
def check(name, ok, detail="", critical=True):
    checks.append((name, bool(ok), detail, critical))

n_tr, n_te = int((df.split == "train").sum()), int((df.split == "test").sum())
check("196 classes in the name list", len(class_names) == 196, f"{len(class_names)}")
check("train images = 8144", n_tr == 8144, f"{n_tr}", critical=not SMOKE)
check("test images = 8041", n_te == 8041, f"{n_te}", critical=not SMOKE)
check("every row found its image file", df.path.notna().all(), f"{int(df.path.isna().sum())} missing")
check("class ids within 0..195", df.class_id.between(0, 195).all(), f"min {df.class_id.min()} max {df.class_id.max()}")
for s in ("train", "test"):
    present = df[df.split == s].class_id.nunique()
    check(f"all 196 classes present in {s}", present == 196, f"{present}", critical=not SMOKE)
per = df[df.split == "test"].class_id.value_counts()
check("test class sizes look like the original (about 24 to 68 per class)", SMOKE or (per.min() >= 15 and per.max() <= 80), f"min {per.min()} max {per.max()}", critical=False)

# Does each image's folder agree with its labelled class? A strong check that labels match the files.
if df.path.notna().all():
    folder = df.path.map(lambda p: Path(p).parent.name)
    in_class_folder = folder.map(norm_name).isin({norm_name(n) for n in class_names})
    if in_class_folder.mean() > 0.9:
        agree = (folder.map(norm_name) == df.class_id.map(lambda i: norm_name(class_names[i])))
        for s in ("train", "test"):
            rate = agree[df.split == s].mean()
            check(f"{s} labels agree with their class folder", rate > 0.99, f"{rate*100:.2f}% agree")

# Boxes and decoding on a sample
sample = df[df.path.notna()].sample(min(200, len(df)), random_state=0)
bad_decode, bad_box = 0, 0
for r in sample.itertuples():
    try:
        with Image.open(r.path) as im:
            w, h = im.size
            im.convert("RGB").load()
        if not np.isnan(r.x1) and not (0 <= r.x1 < r.x2 <= w + 1 and 0 <= r.y1 < r.y2 <= h + 1):
            bad_box += 1
    except Exception:
        bad_decode += 1
check("sample of 200 images decodes", bad_decode == 0, f"{bad_decode} failed")
check("sample boxes lie inside their images", bad_box == 0, f"{bad_box} outside", critical=False)

# The class order must match the backend, because the app reports the class by its index.
BACKEND_URL = "https://raw.githubusercontent.com/sinchanaa23/Apex3D/backend-c/backend/data/stanford_cars_classes.py"
backend_names, remap = None, None
try:
    import urllib.request
    src = urllib.request.urlopen(BACKEND_URL, timeout=20).read().decode()
    ns = {}
    exec(src.split("\ndef ")[0], ns)
    backend_names = ns["STANFORD_CARS_CLASSES"]
except Exception as e:
    print("(could not fetch the backend's class list:", type(e).__name__, ") the class order check is skipped")
if backend_names:
    if [norm_name(n) for n in backend_names] == [norm_name(n) for n in class_names]:
        check("class order matches the backend's list", True, "identical")
    elif sorted(map(norm_name, backend_names)) == sorted(map(norm_name, class_names)):
        by_name = {norm_name(n): i for i, n in enumerate(backend_names)}
        remap = np.array([by_name[norm_name(n)] for n in class_names])
        check("class order matches the backend's list", True, "same classes, different order. Labels will be remapped to the backend's order", critical=False)
    else:
        mine, theirs = {norm_name(n): n for n in class_names}, {norm_name(n): n for n in backend_names}
        only_data = [mine[k] for k in mine if k not in theirs]
        only_backend = [theirs[k] for k in theirs if k not in mine]
        check("class names match the backend's list", False,
              f"dataset has {len(class_names)} names, backend has {len(backend_names)}. Only in the dataset: {only_data[:6]}. Only in the backend: {only_backend[:6]}. "
              "Tell Person C: the backend's list must have exactly the dataset's 196 names in the dataset's order", critical=False)

print(f"{'check':68s} result")
for name, ok, detail, crit in checks:
    print(f"{'PASS' if ok else ('FAIL' if crit else 'WARN'):5s}{name:63s} {detail}")
if any((not ok) and crit for _, ok, _, crit in checks):
    raise SystemExit("A critical data check failed (see FAIL above). Fix this before training. Don't train on data you haven't verified.")
if remap is not None:
    df["class_id"] = remap[df["class_id"].values]
    class_names = list(backend_names)
print("\nData checks passed.")

In [ ]:
# 5. Split. 85/15 stratified train/validation from the 8,144 training images. The test set is only used for final reporting. ----------
from sklearn.model_selection import train_test_split

tr_all = df[df.split == "train"].reset_index(drop=True)
tr_idx, va_idx = train_test_split(np.arange(len(tr_all)), test_size=CFG["val_fraction"], stratify=tr_all.class_id, random_state=CFG["seed"])
train_df = tr_all.iloc[tr_idx].reset_index(drop=True)
val_df = tr_all.iloc[va_idx].reset_index(drop=True)
test_df = df[df.split == "test"].reset_index(drop=True)
print(f"train {len(train_df)}  val {len(val_df)}  test {len(test_df)}")

pd.concat([train_df.assign(part="train"), val_df.assign(part="val"), test_df.assign(part="test")])[["part", "fname", "class_id"]].to_csv(ART / "split.csv", index=False)
json.dump(class_names, open(ART / "classes.json", "w"), indent=1)

# Cache cropped, smaller copies once. Training then reads small files, which keeps the GPU busy on Colab's 2 CPU cores.
def crop_box(im, r, margin):
    if np.isnan(r.x1):
        return im
    w, h = im.size
    bw, bh = r.x2 - r.x1, r.y2 - r.y1
    return im.crop((max(0, int(r.x1 - margin * bw)), max(0, int(r.y1 - margin * bh)), min(w, int(r.x2 + margin * bw)), min(h, int(r.y2 + margin * bh))))

def cache_one(r):
    out = CACHE / f"{r.split}_{hashlib.md5(str(r.path).encode()).hexdigest()[:12]}.jpg"
    if not out.exists():
        with Image.open(r.path) as im:
            im = crop_box(im.convert("RGB"), r, CFG["bbox_margin"])
            s = CFG["cache_short_side"] / min(im.size)
            if s < 1:
                im = im.resize((round(im.width * s), round(im.height * s)), Image.BICUBIC)
            im.save(out, quality=95)
    return str(out)

for d in (train_df, val_df, test_df):
    with ThreadPoolExecutor(8) as ex:
        d["crop"] = list(tqdm(ex.map(cache_one, d.itertuples()), total=len(d), desc="caching crops"))

## 3. Model and data pipeline

- **Network:** ConvNeXt-Tiny from `timm`, pretrained on ImageNet-22k and fine-tuned on ImageNet-1k, with a new 196-way head.
- **Why 224 px:** the backend resizes every crop to 224x224, so training at 224 keeps the app and the notebook consistent. Checking 384 px later means changing the backend too.
- **Augmentation:** random resized crop, flip, RandAugment and random erasing. Random erasing also helps with occlusion.
- **Training:** staged. First only the new head trains, then the whole network trains with layer-wise learning-rate decay, AdamW, a cosine schedule, label smoothing and mixed precision, with early stopping on validation Top-1.

In [ ]:
# 6. Data pipeline and model ---------------------------------------------------------------------------------------
import timm
from timm.optim import create_optimizer_v2
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)    # same as the backend
S = CFG["img_size"]
train_tf = T.Compose([T.RandomResizedCrop(S, scale=(0.4, 1.0), ratio=(0.75, 1.333)), T.RandomHorizontalFlip(),
                      T.RandAugment(num_ops=2, magnitude=9), T.ToTensor(), T.Normalize(MEAN, STD), T.RandomErasing(p=0.25)])
eval_tf = T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(MEAN, STD)])     # squash resize, as in the backend


class Cars(Dataset):
    def __init__(self, frame, tf):
        self.paths, self.y, self.tf = frame["crop"].tolist(), frame["class_id"].astype(int).tolist(), tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            return self.tf(im.convert("RGB")), self.y[i]


def loader(frame, train):
    return DataLoader(Cars(frame, train_tf if train else eval_tf), batch_size=CFG["batch_size"], shuffle=train, drop_last=train,
                      num_workers=CFG["num_workers"], pin_memory=device.type == "cuda", persistent_workers=train and CFG["num_workers"] > 0)

train_loader, val_loader, test_loader = loader(train_df, True), loader(val_df, False), loader(test_df, False)

cfg0 = timm.models.get_pretrained_cfg(CFG["model_name"])
print("pretrained weights:", CFG["model_name"], "| licence listed by timm:", getattr(cfg0, "license", None) or "not listed. Check the model card before publishing")
model = timm.create_model(CFG["model_name"], pretrained=True, num_classes=196, drop_path_rate=CFG["drop_path"]).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f} M")


@torch.no_grad()
def predict_logits(m, dl, desc="eval"):
    m.eval()
    outs, ys = [], []
    for x, y in tqdm(dl, desc=desc, leave=False):
        with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            outs.append(m(x.to(device, non_blocking=True)).float().cpu())
        ys.append(y)
    return torch.cat(outs), torch.cat(ys)

def topk(logits, y, k):
    return (logits.topk(k, dim=1).indices == y[:, None]).any(1).float().mean().item()

In [ ]:
# 7. Training, in two stages, with a checkpoint every epoch so a Colab disconnect costs at most one epoch -----------------------------
use_amp = device.type == "cuda"
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])


def lr_lambda(total_steps, warmup_steps):
    def f(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * p))
    return f


def run_stage(name, epochs, lr, head_only, patience=None):
    last_p, best_p = CKPT / f"{name}_last.pt", CKPT / f"{name}_best.pt"
    for p in model.parameters():
        p.requires_grad = not head_only
    for p in model.get_classifier().parameters():
        p.requires_grad = True
    if head_only:
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=CFG["weight_decay"])
    else:
        opt = create_optimizer_v2(model, "adamw", lr=lr, weight_decay=CFG["weight_decay"], layer_decay=CFG["layer_decay"])
    for g in opt.param_groups:
        g["lr"] = lr * g.get("lr_scale", 1.0)       # layer-wise learning-rate decay (timm stores it as lr_scale)
    rates = [g["lr"] for g in opt.param_groups if any(q.requires_grad for q in g["params"])]
    print(f"[{name}] learning rates across layer groups: {min(rates):.2e} (earliest layers) to {max(rates):.2e} (head)")
    steps = epochs * len(train_loader)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda(steps, CFG["warmup_epochs"] * len(train_loader)))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    state = dict(epoch=0, best=-1.0, bad=0, history=[], done=False)

    if last_p.exists():
        ck = torch.load(last_p, map_location=device, weights_only=False)
        old = ck.get("cfg", {})
        if old.get("model_name", CFG["model_name"]) != CFG["model_name"] or old.get("img_size", S) != S:
            raise SystemExit(f"A saved checkpoint in {CKPT} was made with different settings {old}. Delete that folder to train again from scratch.")
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        state = ck["state"]
        print(f"[{name}] resumed after epoch {state['epoch']}" + (" (finished)" if state["done"] else ""))
    if state["done"]:
        model.load_state_dict(torch.load(best_p, map_location=device, weights_only=False)["model"])
        return state["history"]

    for epoch in range(state["epoch"], epochs):
        model.train(); t0 = time.time(); run_loss, n = 0.0, 0
        for x, y in tqdm(train_loader, desc=f"{name} epoch {epoch+1}/{epochs}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                loss = criterion(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item() * len(y); n += len(y)
        vl, vy = predict_logits(model, val_loader, "val")
        row = dict(stage=name, epoch=epoch + 1, train_loss=run_loss / n, val_loss=F.cross_entropy(vl, vy).item(),
                   val_top1=topk(vl, vy, 1), val_top5=topk(vl, vy, 5), lr=opt.param_groups[-1]["lr"], seconds=time.time() - t0)
        state["history"].append(row); state["epoch"] = epoch + 1
        improved = row["val_top1"] > state["best"]
        if improved:
            state["best"], state["bad"] = row["val_top1"], 0
            torch.save({"model": model.state_dict()}, best_p)
        else:
            state["bad"] += 1
        stop = patience is not None and state["bad"] >= patience
        state["done"] = stop or state["epoch"] >= epochs
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(), "state": state, "cfg": {"model_name": CFG["model_name"], "img_size": S}}, last_p)
        print(f"[{name}] epoch {epoch+1:2d}  loss {row['train_loss']:.3f}  val loss {row['val_loss']:.3f}  top1 {row['val_top1']*100:5.2f}  top5 {row['val_top5']*100:5.2f}  "
              f"{row['seconds']:.0f}s" + ("  *best*" if improved else ""))
        if stop:
            print(f"[{name}] early stop: no improvement for {patience} epochs")
            break
    model.load_state_dict(torch.load(best_p, map_location=device, weights_only=False)["model"])
    return state["history"]


t_start = time.time()
hist = run_stage("stage1_head", CFG["head_epochs"], CFG["head_lr"], head_only=True)
hist += run_stage("stage2_full", CFG["full_epochs"], CFG["full_lr"], head_only=False, patience=CFG["patience"])
print(f"\ntraining finished in {(time.time()-t_start)/60:.1f} min (this session)")
history = pd.DataFrame(hist)
history.to_csv(ART / "train_history.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(history.index + 1, history.train_loss, label="train"); ax[0].plot(history.index + 1, history.val_loss, label="validation")
ax[0].set(title="Loss", xlabel="epoch"); ax[0].legend()
ax[1].plot(history.index + 1, history.val_top1 * 100, label="Top-1"); ax[1].plot(history.index + 1, history.val_top5 * 100, label="Top-5")
ax[1].set(title="Validation accuracy (%)", xlabel="epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig(ART / "training_curves.png", dpi=130); plt.show()

## 4. Evaluation on the unseen test set

The test set is used here for the first time. Results are cached, so running the cell again never lets test results influence decisions made on validation.

In [ ]:
# 8. Validation and test predictions ------------------------------------------------------------------------------------
val_logits, val_y = predict_logits(model, val_loader, "val")
best_file = CKPT / "stage2_full_best.pt"
fingerprint = f"{best_file.stat().st_size}-{int(best_file.stat().st_mtime)}-{CFG['img_size']}"
test_cache = ART / "test_logits.pt"
tc = torch.load(test_cache, weights_only=False) if test_cache.exists() else None
if tc is not None and tc.get("fingerprint") == fingerprint and not os.environ.get("APEX_RERUN_TEST"):
    test_logits, test_y = tc["logits"], tc["y"]
    print("(using the cached test predictions from the first evaluation of this model)")
else:
    if tc is not None:
        print("(a test result for a different model was found, so the test set was evaluated again)")
    test_logits, test_y = predict_logits(model, test_loader, "test")
    torch.save({"logits": test_logits, "y": test_y, "fingerprint": fingerprint}, test_cache)

val_top1, val_top5 = topk(val_logits, val_y, 1), topk(val_logits, val_y, 5)
test_top1, test_top5 = topk(test_logits, test_y, 1), topk(test_logits, test_y, 5)
print(f"validation  Top-1 {val_top1*100:.2f}%   Top-5 {val_top5*100:.2f}%")
print(f"test        Top-1 {test_top1*100:.2f}%   Top-5 {test_top5*100:.2f}%   ({len(test_y)} images, ground-truth crops)")
gap = val_top1 - test_top1
print(f"validation minus test Top-1: {gap*100:+.2f} points")
if gap > 0.10:
    print("WARNING: the test result is much worse than validation. If the dataset's test labels came from a third party, they may be wrong. "
          "Check the test labels before reporting anything.")

# Per-class precision, recall and F1, and the most confused pairs
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
pred = test_logits.argmax(1).numpy(); yt = test_y.numpy()
p, r, f1, support = precision_recall_fscore_support(yt, pred, labels=range(196), zero_division=0)
per_class = pd.DataFrame(dict(class_id=range(196), class_name=class_names, precision=p, recall=r, f1=f1, support=support)).sort_values("f1")
per_class.to_csv(ART / "per_class_metrics.csv", index=False)
print("\nweakest 10 classes by F1:"); display(per_class.head(10).round(3))
print(f"macro F1 {f1.mean():.3f}")

cm = confusion_matrix(yt, pred, labels=range(196))
off = cm.copy(); np.fill_diagonal(off, 0)
pairs = [dict(a=class_names[i], b=class_names[j], count=int(off[i, j])) for i, j in zip(*np.nonzero(off))]
pairs = sorted(pairs, key=lambda d: -d["count"])[:15]
print("\nmost confused pairs (true class, predicted class):"); display(pd.DataFrame(pairs))

plt.figure(figsize=(8, 7)); plt.imshow(np.log1p(cm), cmap="magma"); plt.title("Confusion matrix (log scale)"); plt.xlabel("predicted"); plt.ylabel("true")
plt.colorbar(); plt.tight_layout(); plt.savefig(ART / "confusion_matrix.png", dpi=130); plt.show()

In [ ]:
# 9. Test accuracy when the crop comes from the detector (what the app does), instead of the ground-truth box --------------------------
det_results = None
if CFG["run_detector_crop_eval"]:
    from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights
    import torchvision.transforms.functional as TF
    detector = fasterrcnn_mobilenet_v3_large_fpn(weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT).to(device).eval()
    VEHICLE = {3, 6, 8}   # COCO: car, bus, truck
    orig = test_df[["path", "x1", "y1", "x2", "y2"]].reset_index(drop=True)
    found, det_logits = 0, []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(orig), 8), desc="detect and classify"):
            rows = orig.iloc[i:i + 8]
            ims = [Image.open(pth).convert("RGB") for pth in rows.path]
            outs = detector([TF.to_tensor(im).to(device) for im in ims])
            crops = []
            for im, o in zip(ims, outs):
                keep = [(s_.item(), b.tolist()) for l, s_, b in zip(o["labels"], o["scores"], o["boxes"]) if l.item() in VEHICLE and s_.item() >= 0.5]
                box = max(keep)[1] if keep else None
                if box and box[2] - box[0] >= 4 and box[3] - box[1] >= 4:
                    found += 1
                    crops.append(im.crop(tuple(int(v) for v in box)))
                else:
                    crops.append(im)        # no vehicle found: the backend classifies the whole frame too
            x = torch.stack([eval_tf(c) for c in crops]).to(device)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                det_logits.append(model(x).float().cpu())
    del detector
    det_logits = torch.cat(det_logits)
    det_results = dict(top1=topk(det_logits, test_y, 1), top5=topk(det_logits, test_y, 5), detection_rate=found / len(orig))
    print(f"detector crops: Top-1 {det_results['top1']*100:.2f}%   Top-5 {det_results['top5']*100:.2f}%   (a vehicle was found in {det_results['detection_rate']*100:.1f}% of images)")
else:
    print("(detector-crop evaluation skipped)")

## 5. Confidence: calibration and thresholds

The backend calls a result *confident*, *ambiguous* or *rejected* from its top probability and the gap to the runner-up. Raw softmax scores from a
fine-tuned network are usually over-confident, so **temperature scaling** is fitted on the validation set only, then baked into the exported model.
Thresholds are chosen on validation. The test set is used only to report how they perform.

In [ ]:
# 10. Temperature scaling and confidence analysis ------------------------------------------------------------------------------------
def ece(logits, y, bins=15):
    p = logits.softmax(1); conf, pr = p.max(1); acc = (pr == y).float()
    edges = torch.linspace(0, 1, bins + 1); total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.float().mean().item() * abs(acc[m].mean().item() - conf[m].mean().item())
    return total

log_t = torch.zeros(1, requires_grad=True)
opt_t = torch.optim.LBFGS([log_t], lr=0.05, max_iter=200)
def closure():
    opt_t.zero_grad(); l = F.cross_entropy(val_logits / log_t.exp(), val_y); l.backward(); return l
opt_t.step(closure)
TEMP = float(log_t.exp().item())
print(f"temperature T = {TEMP:.3f}   (T > 1 means the raw scores were over-confident)")
print(f"validation ECE  before {ece(val_logits, val_y):.4f}  after {ece(val_logits / TEMP, val_y):.4f}")
print(f"test       ECE  before {ece(test_logits, test_y):.4f}  after {ece(test_logits / TEMP, test_y):.4f}")

def curve(logits, y, thresholds):
    p = (logits / TEMP).softmax(1); conf, pr = p.max(1); ok = (pr == y)
    rows = []
    for t in thresholds:
        m = conf >= t
        rows.append(dict(threshold=float(t), coverage=m.float().mean().item(), accuracy=ok[m].float().mean().item() if m.any() else None))
    return rows

grid = [round(t, 2) for t in np.arange(0.0, 0.96, 0.05)]
val_curve, test_curve = curve(val_logits, val_y, grid), curve(test_logits, test_y, grid)
chosen = next((r["threshold"] for r in val_curve if r["accuracy"] is not None and r["accuracy"] >= CFG["target_kept_accuracy"]), None)
print(f"\nthreshold chosen on validation: {chosen}  (the lowest at which the kept images reach {CFG['target_kept_accuracy']*100:.0f}% accuracy)" if chosen is not None
      else f"\nno threshold reaches {CFG['target_kept_accuracy']*100:.0f}% accuracy on validation. Report that.")
display(pd.DataFrame(test_curve).round(3).T)

plt.figure(figsize=(6.5, 4))
plt.plot([r["threshold"] for r in test_curve], [r["coverage"] * 100 for r in test_curve], label="coverage (images kept)")
plt.plot([r["threshold"] for r in test_curve if r["accuracy"] is not None], [r["accuracy"] * 100 for r in test_curve if r["accuracy"] is not None], label="accuracy on kept images")
if chosen is not None: plt.axvline(chosen, color="gray", ls=":", label="chosen on validation")
plt.xlabel("confidence threshold"); plt.ylabel("%"); plt.title("Test set: coverage and accuracy"); plt.legend(); plt.tight_layout()
plt.savefig(ART / "confidence_curve.png", dpi=130); plt.show()

# What the backend's current rules (top1 >= 0.50 and margin >= 0.15 is confident, top1 < 0.20 is rejected) would do on the test set
pr_ = (test_logits / TEMP).softmax(1); top2 = pr_.topk(2, 1).values; t1, margin = top2[:, 0], top2[:, 0] - top2[:, 1]
ok_ = (pr_.argmax(1) == test_y)
state = np.where(t1 < 0.20, "rejected", np.where((t1 >= 0.50) & (margin >= 0.15), "confident", "ambiguous"))
rows = [dict(state=s, share=float((state == s).mean()), top1_accuracy=float(ok_[torch.tensor(state == s)].float().mean()) if (state == s).any() else None) for s in ("confident", "ambiguous", "rejected")]
print("\nthe backend's default thresholds on the calibrated test predictions:"); display(pd.DataFrame(rows).round(3))

# Optional: a small folder of non-Stanford vehicles (newer cars, vans, trucks) at WORK/ood_images, to see how often they are rejected
OOD = WORK / "ood_images"
if OOD.exists() and any(OOD.iterdir()):
    files = [f for f in OOD.rglob("*") if f.suffix.lower() in IMG_EXT]
    with torch.no_grad():
        model.eval(); pp = torch.cat([(model(eval_tf(Image.open(f).convert("RGB"))[None].to(device)).float().cpu() / TEMP).softmax(1) for f in files])
    t1o = pp.max(1).values; m2 = pp.topk(2, 1).values
    print(f"\n{len(files)} out-of-taxonomy images (a small qualitative sample): rejected {(t1o < 0.2).float().mean()*100:.0f}%, "
          f"not confident {((m2[:,0] < 0.5) | (m2[:,0]-m2[:,1] < 0.15)).float().mean()*100:.0f}%")
else:
    print(f"\n(Optional: put 20 to 50 photos of vehicles that are not in Stanford Cars in {OOD} to measure how often they are rejected.)")

## 6. Optimisation: size, speed and accuracy before and after

The proposal needs at least one optimisation. This cell exports to **ONNX** (with the temperature baked in, so the backend's softmax gives calibrated
probabilities), applies **INT8 dynamic quantization**, and measures **half precision** on the GPU. Accuracy for fp32 and INT8 is compared on the same
test subset, because CPU inference in Colab is slow.

*Timings are for this Colab machine's CPU and GPU, not for your final host. Re-measure there.*

In [ ]:
# 11. Export and optimise -------------------------------------------------------------------------------------------------
import onnx, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

model_cpu = timm.create_model(CFG["model_name"], pretrained=False, num_classes=196).eval()
model_cpu.load_state_dict({k: v.cpu() for k, v in model.state_dict().items()})

class Calibrated(nn.Module):
    """Logits divided by the fitted temperature, so softmax(output) is calibrated."""
    def __init__(self, m, t):
        super().__init__(); self.m = m; self.register_buffer("inv_t", torch.tensor(1.0 / t))
    def forward(self, x):
        return self.m(x) * self.inv_t

wrapped = Calibrated(model_cpu, TEMP).eval()
onnx_path, int8_path = ART / "model.onnx", ART / "model_quantized.onnx"
dummy = torch.randn(1, 3, S, S)
kw = dict(input_names=["input"], output_names=["logits"], dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17)
try:
    torch.onnx.export(wrapped, dummy, str(onnx_path), dynamo=False, **kw)
except TypeError:
    torch.onnx.export(wrapped, dummy, str(onnx_path), **kw)
onnx.checker.check_model(str(onnx_path))

# The exported file must give the same answers as the PyTorch model
xs = torch.stack([eval_tf(Image.open(p).convert("RGB")) for p in val_df["crop"][:8]])
with torch.no_grad():
    ref = wrapped(xs).numpy()
sess32 = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
diff = float(np.abs(sess32.run(None, {"input": xs.numpy()})[0] - ref).max())
print(f"ONNX vs PyTorch, largest logit difference on 8 images: {diff:.5f}")
assert diff < 1e-2, "The exported model does not match the PyTorch model"

int8_ok = True
try:
    quantize_dynamic(str(onnx_path), str(int8_path), weight_type=QuantType.QInt8, extra_options={"DefaultTensorType": onnx.TensorProto.FLOAT})
except Exception as e:
    int8_ok = False
    print("INT8 quantization failed:", type(e).__name__, str(e)[:200], "\nThe report will say so. Half precision below is the fallback.")

mb = lambda p: Path(p).stat().st_size / 1e6
torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()}, "class_names": class_names, "temperature": TEMP,
            "model_name": CFG["model_name"], "img_size": S, "mean": MEAN, "std": STD}, ART / "convnext_stanford_cars_state.pth")
sizes = dict(fp32_pth_mb=mb(ART / "convnext_stanford_cars_state.pth"), onnx_fp32_mb=mb(onnx_path), onnx_int8_mb=mb(int8_path) if int8_ok else None)
print("sizes (MB):", {k: (round(v, 1) if v else v) for k, v in sizes.items()})

# Accuracy on one shared test subset
rng = np.random.RandomState(0)
sub = rng.choice(len(test_df), min(CFG["quant_eval_n"], len(test_df)), replace=False)
sub_x = torch.stack([eval_tf(Image.open(test_df["crop"][i]).convert("RGB")) for i in sub]).numpy()
sub_y = test_y[sub].numpy()

def ort_top1(sess):
    out = np.concatenate([sess.run(None, {"input": sub_x[i:i + 16]})[0] for i in range(0, len(sub_x), 16)])
    return float((out.argmax(1) == sub_y).mean())
opts = ort.SessionOptions(); opts.intra_op_num_threads = os.cpu_count()
acc = dict(pytorch_fp32=float((test_logits[sub].argmax(1).numpy() == sub_y).mean()), onnx_fp32=ort_top1(ort.InferenceSession(str(onnx_path), opts, providers=["CPUExecutionProvider"])))
if int8_ok:
    try:
        sess8 = ort.InferenceSession(str(int8_path), opts, providers=["CPUExecutionProvider"]); acc["onnx_int8"] = ort_top1(sess8)
    except Exception as e:
        int8_ok = False
        print("The INT8 model could not be run:", type(e).__name__, str(e)[:200], "\nThe report will say so. Half precision is the fallback.")
print(f"Top-1 on the same {len(sub)} test images:", {k: round(v * 100, 2) for k, v in acc.items()})

# Speed, one image at a time
def timeit(fn, runs=CFG["bench_runs"], warm=3):
    for _ in range(warm): fn()
    ts = []
    for _ in range(runs):
        t0 = time.perf_counter(); fn(); ts.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ts))
one = sub_x[:1]; one_t = torch.from_numpy(one)
def cpu_torch():
    with torch.no_grad():
        return wrapped(one_t)
speed = dict(cpu_pytorch_fp32_ms=timeit(cpu_torch), cpu_onnx_fp32_ms=timeit(lambda: sess32.run(None, {"input": one})[0]))
if int8_ok:
    speed["cpu_onnx_int8_ms"] = timeit(lambda: sess8.run(None, {"input": one})[0])
if device.type == "cuda":
    g32 = model.eval(); xg = one_t.to(device)
    def gpu(m, x, half):
        def f():
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=half):
                m(x)
            torch.cuda.synchronize()
        return f
    speed["gpu_fp32_ms"] = timeit(gpu(g32, xg, False)); speed["gpu_fp16_ms"] = timeit(gpu(g32, xg, True))
print("median ms per image:", {k: round(v, 1) for k, v in speed.items()})
print(f"(CPU: {platform.processor() or 'unknown'}, {os.cpu_count()} threads)")

## 7. Save everything for the team

Writes `metrics.json` (read by the app's Research Lab through the backend), the ONNX models the backend loads, and the supporting tables and plots.
It **merges** into an existing `metrics.json`, so results from Person B's notebooks are never overwritten.

In [ ]:
# 12. Metrics file and downloads ------------------------------------------------------------------------------------------------
from datetime import datetime, timezone
now = datetime.now(timezone.utc).isoformat(timespec="seconds")
metrics = {
    "generated_at": now,
    "model_name": f"{CFG['model_name']} fine-tuned on Stanford Cars",
    "test_images": int(len(test_y)),
    "test_top1_acc": test_top1, "test_top5_acc": test_top5,
    "val_top1_acc": val_top1, "val_top5_acc": val_top5,
    "macro_f1": float(f1.mean()), "temperature": TEMP,
    "model_size_mb": sizes["onnx_fp32_mb"],
    "inference_time_cpu_ms": speed["cpu_onnx_fp32_ms"],
    "confidence": {"chosen_threshold": chosen, "rows": test_curve},
    "confused_pairs": {"rows": pairs},
    "optimisation_detail": {"sizes_mb": sizes, "top1_on_shared_test_subset": acc, "subset_size": int(len(sub)), "median_ms_per_image": speed,
                            "note": "timings measured on the Colab machine used for training"},
}
if det_results:
    metrics.update(test_top1_acc_detector_crop=det_results["top1"], test_top5_acc_detector_crop=det_results["top5"], detector_detection_rate=det_results["detection_rate"])
if int8_ok:
    metrics.update(quantized_size_mb=sizes["onnx_int8_mb"], inference_time_quantized_cpu_ms=speed["cpu_onnx_int8_ms"], quantized_top1_on_subset=acc["onnx_int8"])
if "gpu_fp32_ms" in speed:
    metrics["inference_time_gpu_ms"] = speed["gpu_fp32_ms"]; metrics["inference_time_gpu_fp16_ms"] = speed["gpu_fp16_ms"]

json.dump(metrics, open(ART / "metrics_classifier.json", "w"), indent=1)
merged_path = ART / "metrics.json"
merged = json.load(open(merged_path)) if merged_path.exists() else {}
merged.update(metrics)
json.dump(merged, open(merged_path, "w"), indent=1)

json.dump(dict(config=CFG, smoke=SMOKE, finished=now, torch=torch.__version__, timm=timm.__version__, python=platform.python_version(),
               gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else None, dataset_layout=layout, seed=CFG["seed"]), open(ART / "run_info.json", "w"), indent=1)

# Candidate failure cases for Person B to explain in the report (confident mistakes). The explanation must be written by a person.
conf_all = (test_logits / TEMP).softmax(1).max(1).values
wrong = np.nonzero(pred != yt)[0]
worst = wrong[np.argsort(-conf_all[wrong].numpy())][:6]
if len(worst):
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for a, i in zip(axes.ravel(), worst):
        a.imshow(Image.open(test_df["crop"][i])); a.axis("off")
        a.set_title(f"true: {class_names[yt[i]]}\npredicted: {class_names[pred[i]]} ({conf_all[i]*100:.0f}%)", fontsize=8)
    plt.tight_layout(); plt.savefig(ART / "failure_candidates.png", dpi=130); plt.show()

print("\nSaved to", ART)
for f in sorted(ART.iterdir()):
    print(f"  {f.name:40s} {f.stat().st_size/1e6:8.2f} MB")
if IN_COLAB:
    handover = Path("/content/apex3d_handover")
    shutil.rmtree(handover, ignore_errors=True); handover.mkdir()
    for f in ART.iterdir():
        if f.suffix not in (".pth", ".pt"):          # the big training files stay in Drive
            shutil.copy(f, handover / f.name)
    shutil.make_archive("/content/apex3d_handover", "zip", handover)
    print("Everything is also in your Drive at", ART)
    from google.colab import files
    files.download("/content/apex3d_handover.zip")

In [ ]:
# 13. Summary --------------------------------------------------------------------------------------------------------------------
rows = [("Top-1, test (ground-truth crop)", f"{test_top1*100:.2f}%"), ("Top-5, test (ground-truth crop)", f"{test_top5*100:.2f}%")]
if det_results:
    rows += [("Top-1, test (detector crop)", f"{det_results['top1']*100:.2f}%"), ("Top-5, test (detector crop)", f"{det_results['top5']*100:.2f}%")]
rows += [("macro F1, test", f"{f1.mean():.3f}"), ("temperature", f"{TEMP:.3f}"), ("model size, ONNX fp32", f"{sizes['onnx_fp32_mb']:.1f} MB")]
if int8_ok:
    rows += [("model size, ONNX INT8", f"{sizes['onnx_int8_mb']:.1f} MB"), ("Top-1 fp32 vs INT8, same subset", f"{acc['onnx_fp32']*100:.2f}% vs {acc['onnx_int8']*100:.2f}%")]
for k, v in rows:
    print(f"{k:38s} {v}")
if SMOKE:
    print("\n*** SMOKE TEST: these numbers mean nothing ***")
print("""
Hand over to the team
  1. Copy model.onnx and model_quantized.onnx to the repo's  checkpoints/  folder.
  2. Copy metrics.json to  notebooks/metrics.json  (the backend reads it from there).
  3. Person C: add  onnxruntime  to requirements.txt, and confirm the classes.json order matches backend/data/stanford_cars_classes.py.
  4. Write the three failure-case explanations in the evaluation report. The plot above only suggests candidates.
""")